In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

query = """
SELECT *
FROM read_csv_auto('../data/raw/titanic.csv')
LIMIT 5
"""

df_preview = con.execute(query).df()
df_preview

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


In [2]:
con.execute("""
CREATE OR REPLACE VIEW titanic AS
SELECT *
FROM read_csv_auto('../data/raw/titanic.csv')
""")

print("Vista 'titanic' creada correctamente")

Vista 'titanic' creada correctamente


In [3]:
con.execute("SELECT * FROM titanic LIMIT 10").df()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,None,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,None,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,None,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,None,C


In [4]:
con.execute("SELECT COUNT(*) AS total_filas FROM titanic").df()

,total_filas
0,891


In [5]:
con.execute("""
SELECT
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS nulos_age,
    SUM(CASE WHEN Cabin IS NULL THEN 1 ELSE 0 END) AS nulos_cabin,
    SUM(CASE WHEN Embarked IS NULL THEN 1 ELSE 0 END) AS nulos_embarked
FROM titanic
""").df()

,nulos_age,nulos_cabin,nulos_embarked
0,177.0,687.0,2.0


In [6]:
con.execute("""
SELECT
    AVG(Survived) AS tasa_supervivencia
FROM titanic
""").df()

,tasa_supervivencia
0,0.383838


In [7]:
con.execute("""
SELECT
    Sex,
    AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Sex
ORDER BY tasa_supervivencia DESC
""").df()

,Sex,tasa_supervivencia
0,female,0.742038
1,male,0.188908


In [8]:
con.execute("""
SELECT
    Pclass,
    AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""").df()

,Pclass,tasa_supervivencia
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [9]:
con.execute("""
SELECT
    Pclass,
    AVG(Fare) AS tarifa_promedio
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""").df()

,Pclass,tarifa_promedio
0,1,84.154687
1,2,20.662183
2,3,13.675550


In [10]:
con.execute("""
SELECT
    Sex,
    Pclass,
    AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
""").df()

,Sex,Pclass,tasa_supervivencia
0,female,1,0.968085
1,female,2,0.921053
2,female,3,0.500000
3,male,1,0.368852
4,male,2,0.157407
5,male,3,0.135447


In [11]:
con.execute("""
SELECT
    Name, Sex, Pclass, Fare, Survived
FROM titanic
WHERE Sex = 'female' AND Pclass = 1
LIMIT 10
""").df()

,Name,Sex,Pclass,Fare,Survived
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,1,71.2833,1
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,1,53.1000,1
2,"Bonnell, Miss. Elizabeth",female,1,26.5500,1
3,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,1,146.5208,1
4,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,1,76.7292,1
5,"Icard, Miss. Amelie",female,1,80.0000,1
6,"Fortune, Miss. Mabel Helen",female,1,263.0000,1
7,"Newsom, Miss. Helen Monypeny",female,1,26.2833,1
8,"Pears, Mrs. Thomas (Edith Wearne)",female,1,66.6000,1
9,"Chibnall, Mrs. (Edith Martha Bowerman)",female,1,55.0000,1


In [12]:
con.execute("""
SELECT
    Name, Sex, Pclass, Age, Survived
FROM titanic
WHERE Sex = 'male' AND Pclass = 3 AND Survived = 0
LIMIT 10
""").df()

,Name,Sex,Pclass,Age,Survived
0,"Braund, Mr. Owen Harris",male,3,22.0,0
1,"Allen, Mr. William Henry",male,3,35.0,0
2,"Moran, Mr. James",male,3,NaN,0
3,"Palsson, Master. Gosta Leonard",male,3,2.0,0
4,"Saundercock, Mr. William Henry",male,3,20.0,0
5,"Andersson, Mr. Anders Johan",male,3,39.0,0
6,"Rice, Master. Eugene",male,3,2.0,0
7,"Emir, Mr. Farred Chehab",male,3,NaN,0
8,"Todoroff, Mr. Lalio",male,3,NaN,0
9,"Cann, Mr. Ernest Charles",male,3,21.0,0


In [13]:
con.execute("""
SELECT
    Name, Age, Pclass, Sex, Survived
FROM titanic
WHERE Age > 60
ORDER BY Age DESC
""").df()

,Name,Age,Pclass,Sex,Survived
0,"Barkworth, Mr. Algernon Henry Wilson",80.0,1,male,1
1,"Svensson, Mr. Johan",74.0,3,male,0
2,"Goldschmidt, Mr. George B",71.0,1,male,0
3,"Artagaveytia, Mr. Ramon",71.0,1,male,0
4,"Connors, Mr. Patrick",70.5,3,male,0
5,"Mitchell, Mr. Henry Michael",70.0,2,male,0
6,"Crosby, Capt. Edward Gifford",70.0,1,male,0
7,"Wheadon, Mr. Edward H",66.0,2,male,0
8,"Ostby, Mr. Engelhart Cornelius",65.0,1,male,0
9,"Duane, Mr. Frank",65.0,3,male,0


In [14]:
con.execute("""
SELECT
    PassengerId,
    Name,
    Age,
    CASE
        WHEN Age <= 12 THEN 'Niño'
        WHEN Age <= 18 THEN 'Adolescente'
        WHEN Age <= 35 THEN 'Joven Adulto'
        WHEN Age <= 60 THEN 'Adulto'
        ELSE 'Adulto Mayor'
    END AS grupo_edad
FROM titanic
WHERE Age IS NOT NULL
LIMIT 20
""").df()

,PassengerId,Name,Age,grupo_edad
0,1,"Braund, Mr. Owen Harris",22.0,Joven Adulto
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adulto
2,3,"Heikkinen, Miss. Laina",26.0,Joven Adulto
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Joven Adulto
4,5,"Allen, Mr. William Henry",35.0,Joven Adulto
5,7,"McCarthy, Mr. Timothy J",54.0,Adulto
6,8,"Palsson, Master. Gosta Leonard",2.0,Niño
7,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,Joven Adulto
8,10,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,Adolescente
9,11,"Sandstrom, Miss. Marguerite Rut",4.0,Niño


In [15]:
con.execute("""
SELECT
    CASE
        WHEN Age <= 12 THEN 'Niño'
        WHEN Age <= 18 THEN 'Adolescente'
        WHEN Age <= 35 THEN 'Joven Adulto'
        WHEN Age <= 60 THEN 'Adulto'
        ELSE 'Adulto Mayor'
    END AS grupo_edad,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
WHERE Age IS NOT NULL
GROUP BY 1
ORDER BY tasa_supervivencia DESC
""").df()

,grupo_edad,tasa_supervivencia,total_pasajeros
0,Niño,0.579710,69
1,Adolescente,0.428571,70
2,Adulto,0.400000,195
3,Joven Adulto,0.382682,358
4,Adulto Mayor,0.227273,22


In [16]:
con.execute("""
SELECT
    PassengerId,
    Name,
    SibSp,
    Parch,
    SibSp + Parch + 1 AS tamano_familia
FROM titanic
LIMIT 10
""").df()

,PassengerId,Name,SibSp,Parch,tamano_familia
0,1,"Braund, Mr. Owen Harris",1,0,2
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,2
2,3,"Heikkinen, Miss. Laina",0,0,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,2
4,5,"Allen, Mr. William Henry",0,0,1
5,6,"Moran, Mr. James",0,0,1
6,7,"McCarthy, Mr. Timothy J",0,0,1
7,8,"Palsson, Master. Gosta Leonard",3,1,5
8,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",0,2,3
9,10,"Nasser, Mrs. Nicholas (Adele Achem)",1,0,2


In [17]:
con.execute("""
SELECT
    SibSp + Parch + 1 AS tamano_familia,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
GROUP BY 1
ORDER BY tamano_familia
""").df()

,tamano_familia,tasa_supervivencia,total_pasajeros
0,1,0.303538,537
1,2,0.552795,161
2,3,0.578431,102
3,4,0.724138,29
4,5,0.200000,15
5,6,0.136364,22
6,7,0.333333,12
7,8,0.000000,6
8,11,0.000000,7


In [18]:
con.execute("""
SELECT
    Name,
    Pclass,
    Sex,
    Fare,
    Survived
FROM titanic
ORDER BY Fare DESC
LIMIT 10
""").df()

,Name,Pclass,Sex,Fare,Survived
0,"Cardeza, Mr. Thomas Drake Martinez",1,male,512.3292,1
1,"Ward, Miss. Anna",1,female,512.3292,1
2,"Lesurer, Mr. Gustave J",1,male,512.3292,1
3,"Fortune, Miss. Mabel Helen",1,female,263.0000,1
4,"Fortune, Miss. Alice Elizabeth",1,female,263.0000,1
5,"Fortune, Mr. Mark",1,male,263.0000,0
6,"Fortune, Mr. Charles Alexander",1,male,263.0000,0
7,"Ryerson, Miss. Emily Borie",1,female,262.3750,1
8,"Ryerson, Miss. Susan Parker ""Suzette""",1,female,262.3750,1
9,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",1,female,247.5208,1


In [19]:
con.execute("""
SELECT
    Embarked,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
GROUP BY Embarked
ORDER BY tasa_supervivencia DESC
""").df()

,Embarked,tasa_supervivencia,total_pasajeros
0,None,1.000000,2
1,C,0.553571,168
2,Q,0.389610,77
3,S,0.336957,644


In [20]:
con.execute("""
SELECT
    Sex,
    AVG(Age) AS edad_promedio
FROM titanic
GROUP BY Sex
""").df()

,Sex,edad_promedio
0,female,27.915709
1,male,30.726645


In [21]:
con.execute("SELECT COUNT(*) AS total_filas FROM titanic").df()

con.execute("""
SELECT Sex, AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Sex
ORDER BY tasa_supervivencia DESC
""").df()

con.execute("""
SELECT Pclass, AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""").df()

con.execute("""
SELECT Sex, Pclass, AVG(Survived) AS tasa_supervivencia
FROM titanic
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
""").df()

,Sex,Pclass,tasa_supervivencia
0,female,1,0.968085
1,female,2,0.921053
2,female,3,0.500000
3,male,1,0.368852
4,male,2,0.157407
5,male,3,0.135447


In [22]:
con.execute("""
SELECT
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS nulos_age,
    SUM(CASE WHEN Cabin IS NULL THEN 1 ELSE 0 END) AS nulos_cabin,
    SUM(CASE WHEN Embarked IS NULL THEN 1 ELSE 0 END) AS nulos_embarked
FROM titanic
""").df()

,nulos_age,nulos_cabin,nulos_embarked
0,177.0,687.0,2.0


In [23]:
con.execute("""
SELECT
    Embarked,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
GROUP BY Embarked
ORDER BY tasa_supervivencia DESC
""").df()

,Embarked,tasa_supervivencia,total_pasajeros
0,None,1.000000,2
1,C,0.553571,168
2,Q,0.389610,77
3,S,0.336957,644


In [24]:
con.execute("""
SELECT
    Sex,
    AVG(Age) AS edad_promedio
FROM titanic
GROUP BY Sex
ORDER BY edad_promedio DESC
""").df()

,Sex,edad_promedio
0,male,30.726645
1,female,27.915709


In [25]:
con.execute("""
SELECT
    Name,
    Pclass,
    Sex,
    Fare,
    Survived
FROM titanic
ORDER BY Fare DESC
LIMIT 10
""").df()

,Name,Pclass,Sex,Fare,Survived
0,"Cardeza, Mr. Thomas Drake Martinez",1,male,512.3292,1
1,"Ward, Miss. Anna",1,female,512.3292,1
2,"Lesurer, Mr. Gustave J",1,male,512.3292,1
3,"Fortune, Miss. Mabel Helen",1,female,263.0000,1
4,"Fortune, Miss. Alice Elizabeth",1,female,263.0000,1
5,"Fortune, Mr. Mark",1,male,263.0000,0
6,"Fortune, Mr. Charles Alexander",1,male,263.0000,0
7,"Ryerson, Miss. Emily Borie",1,female,262.3750,1
8,"Ryerson, Miss. Susan Parker ""Suzette""",1,female,262.3750,1
9,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",1,female,247.5208,1


In [26]:
con.execute("""
SELECT
    Name,
    Pclass,
    Sex,
    Fare,
    Survived
FROM titanic
WHERE Sex = 'female' AND Pclass = 1
LIMIT 10
""").df()

,Name,Pclass,Sex,Fare,Survived
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,71.2833,1
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,53.1000,1
2,"Bonnell, Miss. Elizabeth",1,female,26.5500,1
3,"Spencer, Mrs. William Augustus (Marie Eugenie)",1,female,146.5208,1
4,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",1,female,76.7292,1
5,"Icard, Miss. Amelie",1,female,80.0000,1
6,"Fortune, Miss. Mabel Helen",1,female,263.0000,1
7,"Newsom, Miss. Helen Monypeny",1,female,26.2833,1
8,"Pears, Mrs. Thomas (Edith Wearne)",1,female,66.6000,1
9,"Chibnall, Mrs. (Edith Martha Bowerman)",1,female,55.0000,1


In [27]:
con.execute("""
SELECT
    Name,
    Pclass,
    Sex,
    Age,
    Survived
FROM titanic
WHERE Sex = 'male' AND Pclass = 3 AND Survived = 0
LIMIT 10
""").df()

,Name,Pclass,Sex,Age,Survived
0,"Braund, Mr. Owen Harris",3,male,22.0,0
1,"Allen, Mr. William Henry",3,male,35.0,0
2,"Moran, Mr. James",3,male,NaN,0
3,"Palsson, Master. Gosta Leonard",3,male,2.0,0
4,"Saundercock, Mr. William Henry",3,male,20.0,0
5,"Andersson, Mr. Anders Johan",3,male,39.0,0
6,"Rice, Master. Eugene",3,male,2.0,0
7,"Emir, Mr. Farred Chehab",3,male,NaN,0
8,"Todoroff, Mr. Lalio",3,male,NaN,0
9,"Cann, Mr. Ernest Charles",3,male,21.0,0


In [28]:
con.execute("""
SELECT
    Name,
    Age,
    Pclass,
    Sex,
    Survived
FROM titanic
WHERE Age > 60
ORDER BY Age DESC
""").df()

,Name,Age,Pclass,Sex,Survived
0,"Barkworth, Mr. Algernon Henry Wilson",80.0,1,male,1
1,"Svensson, Mr. Johan",74.0,3,male,0
2,"Goldschmidt, Mr. George B",71.0,1,male,0
3,"Artagaveytia, Mr. Ramon",71.0,1,male,0
4,"Connors, Mr. Patrick",70.5,3,male,0
5,"Mitchell, Mr. Henry Michael",70.0,2,male,0
6,"Crosby, Capt. Edward Gifford",70.0,1,male,0
7,"Wheadon, Mr. Edward H",66.0,2,male,0
8,"Ostby, Mr. Engelhart Cornelius",65.0,1,male,0
9,"Duane, Mr. Frank",65.0,3,male,0


## Conclusiones del análisis SQL

- SQL permitió responder las mismas preguntas de negocio trabajadas previamente con pandas.
- Se confirmaron diferencias importantes de supervivencia según sexo, clase, edad y tamaño de familia.
- La combinación de sexo y clase muestra patrones especialmente marcados: las mujeres de primera y segunda clase presentan las tasas más altas, mientras que los hombres de tercera clase muestran las más bajas.
- DuckDB permite consultar directamente archivos CSV de manera eficiente, lo que facilita análisis exploratorios sin necesidad de montar una base de datos tradicional.

In [29]:
con.execute("""
SELECT
    CASE
        WHEN Age <= 12 THEN 'Niño'
        WHEN Age <= 18 THEN 'Adolescente'
        WHEN Age <= 35 THEN 'Joven Adulto'
        WHEN Age <= 60 THEN 'Adulto'
        ELSE 'Adulto Mayor'
    END AS grupo_edad,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
WHERE Age IS NOT NULL
GROUP BY 1
ORDER BY tasa_supervivencia DESC
""").df()

con.execute("""
SELECT
    SibSp + Parch + 1 AS tamano_familia,
    AVG(Survived) AS tasa_supervivencia,
    COUNT(*) AS total_pasajeros
FROM titanic
GROUP BY 1
ORDER BY tamano_familia
""").df()

,tamano_familia,tasa_supervivencia,total_pasajeros
0,1,0.303538,537
1,2,0.552795,161
2,3,0.578431,102
3,4,0.724138,29
4,5,0.200000,15
5,6,0.136364,22
6,7,0.333333,12
7,8,0.000000,6
8,11,0.000000,7


## Conclusiones finales del análisis SQL

- SQL permitió reproducir y validar los principales hallazgos obtenidos previamente con pandas.
- Se confirmaron diferencias relevantes de supervivencia por sexo, clase, edad y tamaño de familia.
- El sexo parece ser el factor más determinante, aunque la clase introduce diferencias adicionales dentro de cada grupo.
- El tamaño de la familia muestra un efecto no lineal: los grupos pequeños y medianos presentan mejores resultados que quienes viajaban solos o en familias muy grandes.
- DuckDB resultó útil para consultar directamente archivos CSV sin necesidad de configurar una base de datos tradicional.